# Logprob Metric Distributions by Text Length

Recompute token-derived metrics using the first N tokens of each text and compare HG/MG distributions in 3D. Metrics are calculated from stored stage-1 top-20 logprobs; no model inference is performed.

In [ ]:
from __future__ import annotations

import os
from IPython.display import display
from pathlib import Path

import seaborn as sns

from logprob_metrics_distributions_utils import (
    TOKEN_DERIVED_METRICS,
    compute_length_metric_table,
    default_length_metrics_path,
    load_length_metric_table,
    plot_length_metric_distributions,
    save_length_metric_table,
)

sns.set_theme(style="whitegrid", context="notebook")

DATA_ROOT = Path(os.environ.get("TEXT_DETECTION_DATA", "D:/Projects/master_diploma/text-detection-data"))
PROCESSED_ROOT = DATA_ROOT / "processed"

## Parameters

Only complete prefixes are evaluated. For example, with `TOKEN_LENGTH_STEP = 50`, a 120-token text contributes lengths 50 and 100.

In [ ]:
selected_datasets = None
selected_models = None
TOKEN_LENGTH_STEP = 50
HISTOGRAM_BINS = 50
RECOMPUTE = False
LENGTH_METRICS_PATH = default_length_metrics_path(PROCESSED_ROOT)

## Recompute Truncated Metrics

In [ ]:
if RECOMPUTE or not LENGTH_METRICS_PATH.exists():
    length_metric_df = compute_length_metric_table(
        PROCESSED_ROOT,
        step=TOKEN_LENGTH_STEP,
        selected_datasets=selected_datasets,
        selected_models=selected_models,
        show_progress=True,
    )
    save_length_metric_table(length_metric_df, LENGTH_METRICS_PATH)
else:
    length_metric_df = load_length_metric_table(LENGTH_METRICS_PATH)

display(length_metric_df.head())
print(f"Rows: {len(length_metric_df):,}")
print(f"Lengths: {sorted(length_metric_df['token_length'].unique()) if not length_metric_df.empty else []}")

## 3D Metric Distributions

Each metric gets one figure per dataset/model pair. The depth axis is the prefix length, and HG/MG are overlaid with separate colors.

In [ ]:
for metric in TOKEN_DERIVED_METRICS:
    if metric in length_metric_df and length_metric_df[metric].notna().any():
        plot_length_metric_distributions(length_metric_df, metric, bins=HISTOGRAM_BINS)
    else:
        print(f"Skipping {metric}: no finite values")